# Notebook 01 — Single-Model Interactive Inspector (debugging aid)

**Goal:** load any ONE of the 20 roster models, walk its module tree, count parameters per submodule, verify the V/F/L/H (or native-schema) extractor worked, apply the freezing policy, and dump a layer-by-layer table — all inline so you can spot-check every number before trusting the batch results in notebook 02.

**When to use this notebook:**
* A model in notebook 02 produced a suspicious param count (e.g. IDEFICS-9B reported language = 0 in Phase-1; that's how the F-3 bug was caught).
* You're adding a new model to the roster and need to verify the extractor for that family finds the right submodules.
* You suspect weight-tying is inflating the total (the LLaVA-1.5 ~152M case — see notebook 02 §12b).
* You want to understand the actual HuggingFace module hierarchy of a model class so you can extend `vlm_audit/components.py`.
* You're chasing a `FLOPs source = manual_estimate` row in notebook 03 and want to see what the extracted module's `forward()` actually looks like.

**Inputs:** one variable — `MODEL_NAME` in §3. Must match a `ModelSpec.name` from `vlm_audit/models.py`.

**Outputs:**
* Inline DataFrames + printed module trees.
* One verbose `.txt` inspection report at `../outputs/logs/nb01_inspect_<sanitized_hf_id>.txt`.
* No measurement JSONs written (this notebook is for inspection only — use notebook 02 for the canonical params record).

**Compute:** GPU helps speed up the load. CPU works. **fp16** is used to match the rest of the pipeline.

**Runtime:** ~30 s (warm cache, small model like ViLT) to ~5 min (cold cache, large model like Pixtral-12B).

**How to read this notebook:** the cells are ordered to take you progressively deeper. §1-2 are setup; §3 is where you pick the model; §4 loads it; §5-11 are the inspection layers. Don't skip §10 (tied-weight detection) — it's the only way to explain LLaVA-1.5's deduped vs raw param counts.

---
## 1. Setup (Runpod)

Standard Runpod boilerplate. If you're not on Runpod, comment out the `!pip install` and provide `HF_TOKEN` however your environment expects.

In [ ]:
# Runpod setup — install deps + ensure HF_TOKEN is in env
%%capture
!pip install -q -r ../requirements.txt

In [ ]:
# Set HF_TOKEN here if not already in env. Required for gated models
# (LLaMA / IDEFICS / Pixtral / Llama-3.2-Vision / InstructBLIP).
import os
# os.environ['HF_TOKEN'] = 'hf_...'  # uncomment + paste your token if needed
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN not set — gated models will fail to download'

In [ ]:
import sys, shutil, gc, time, json, traceback
from pathlib import Path
from io import StringIO
import contextlib

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures'
for d in (MEASUREMENT_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE = Path(os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))).expanduser() / 'hub'

def free_disk_gb(p='/'):
    s = shutil.disk_usage(p)
    return s.free / 1e9, s.total / 1e9

free_gb, total_gb = free_disk_gb()
print(f'Working dir:        {NB_DIR}')
print(f'code_active dir:    {CODE_ACTIVE}')
print(f'vlm_audit:          {CODE_ACTIVE / "vlm_audit"}')
print(f'HF cache dir:       {HF_CACHE}')
print(f'Disk:               {free_gb:.1f} GB free / {total_gb:.1f} GB total')

In [ ]:
import pandas as pd
import numpy as np
import torch

from vlm_audit import (
    MODELS, get_model_list, count_params,
    extract_components, compute_trainable_from_policy,
)
from vlm_audit.runner import _load_model, _resolve_hf_class
from vlm_audit.components import (
    freezing_policy_hint, trainable_components_for_family,
    COMPONENT_NAMES, EXTENDED_COMPONENT_NAMES,
)
from vlm_audit.logging_utils import ExperimentLogger, sanitize_for_filename

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Models in roster: {len(MODELS)}')
print(f'Standard V/F/L/H component names: {COMPONENT_NAMES}')
print(f'Extended (incl. native) component names: {EXTENDED_COMPONENT_NAMES}')

---
## 2. Roster — the 20 valid model names

Before picking a model in §3, scan the roster. Every entry here can be used as `MODEL_NAME`.

In [ ]:
roster_df = pd.DataFrame([
    {
        'name':           m.name,
        'family':         m.family,
        'year':           m.release_year,
        'fp16_GB':        m.fits_in_gb_fp16,
        'native_HW':      f'{m.native_image_hw[0]}x{m.native_image_hw[1]}',
        'recent_gen':     m.recent_gen,
        'trust_remote':   m.requires_trust_remote_code,
        'hf_id':          m.hf_id,
    }
    for m in MODELS
])
print(f'Total models: {len(roster_df)}')
print(f'Families:     {sorted(roster_df.family.unique())} ({len(roster_df.family.unique())} families)')
print()
print('Valid MODEL_NAME values (copy one into §3):')
for n in roster_df['name']:
    print(f'  {n!r}')
roster_df

---
## 3. Choose a model

Set `MODEL_NAME` to one of the names printed above. Re-run from this cell onward to inspect a different model — no need to restart the kernel (cleanup happens in §13).

**Tips for picking:**
* **Quick check:** `ViLT-B/32` (117M, ~460 MB download, loads in ~10s) is the fastest smoke test.
* **Debugging IDEFICS-9B language extractor:** `IDEFICS-9B` (Phase-1 F-3 bug case).
* **Investigating LLaVA-1.5 ~152M discrepancy:** `LLaVA-1.5-7B` (tied lm_head ↔ embed_tokens — see §10).
* **Native fusion (no V/F/L/H):** `Chameleon-7B` to see `fused_transformer` + `embeddings` instead.
* **Decoupled vision:** `Janus-Pro-7B` to see the two `vision_und` + `vision_gen` pathways.
* **Largest:** `Pixtral-12B` (~24 GB fp16) — set `USE_QUANTIZATION = '4bit'` if your GPU is < 30 GB.

Other knobs:
* `USE_QUANTIZATION` — `None` (fp16) / `'int8'` / `'4bit'`. Only quantize for inspection if the model wouldn't otherwise fit.
* `SHOW_FULL_MODULE_TREE` — set to `False` for very large models if the full `print(model)` is too noisy.
* `MODULE_TREE_LINES` — when truncating, how many lines from the head/tail of `str(model)` to show.
* `LEAF_TABLE_TOP_N` — how many leaf modules to keep in the layer-by-layer table (§9). All are saved to the .txt report regardless.

In [ ]:
# === Choose which model to inspect ===
MODEL_NAME            = 'CLIP-ViT-L/14'  # change this — must match a ModelSpec.name above
USE_QUANTIZATION      = None             # None / 'int8' / '4bit' — for big models on small GPUs
SHOW_FULL_MODULE_TREE = True             # set False if str(model) is too long to be useful
MODULE_TREE_LINES     = 40               # head/tail lines to print when truncating
MAX_NAMED_MODULES     = 300              # cap on the named_modules() walk in §6
LEAF_TABLE_TOP_N      = 100              # rows shown in the leaf-module DataFrame in §9
PARAM_DUMP_HEAD_TAIL  = 20               # first/last N entries of named_parameters() in §11

spec = next((m for m in MODELS if m.name == MODEL_NAME), None)
assert spec is not None, (
    f'Unknown model name: {MODEL_NAME!r}. See the roster in §2 for valid names.'
)
print(f'Inspecting: {spec.name}')
print(f'  HF id:           {spec.hf_id}')
print(f'  Family:          {spec.family}')
print(f'  Year:            {spec.release_year}')
print(f'  Native HW:       {spec.native_image_hw}')
print(f'  fp16 GB (~):     {spec.fits_in_gb_fp16}')
print(f'  trust_remote:    {spec.requires_trust_remote_code}')
print(f'  is_moe:          {spec.is_moe}')
print(f'  Quantization:    {USE_QUANTIZATION}')
print(f'  Freezing recipe: {freezing_policy_hint(spec.family)}')
print(f'  Trainable map:   {trainable_components_for_family(spec.family)}')

---
## 4. Load the model + resolve the HF class

Uses the same `_load_model` / `_resolve_hf_class` helpers that the batch runner uses, so what you see here is exactly what notebook 02 will see. The model is loaded in fp16 (matches the rest of the pipeline) unless you set `USE_QUANTIZATION`.

If this cell fails:
* `403 Forbidden` → you haven't accepted the model's license on its HF page, or `HF_TOKEN` is missing.
* `trust_remote_code` errors → the model spec is missing `requires_trust_remote_code=True`.
* `TypeError: from_pretrained() got an unexpected keyword argument 'dtype'` → your `transformers` version is < 4.45; the runner auto-falls-back to `torch_dtype=`.
* `OOM` → try `USE_QUANTIZATION = 'int8'` (halves VRAM) or `'4bit'` (quarters it).

In [ ]:
# Open an inspection log for this run — everything printed below also goes here.
INSPECT_LABEL = f'nb01_inspect__{sanitize_for_filename(spec.hf_id)}'
log = ExperimentLogger(str(LOG_DIR), INSPECT_LABEL)
log.section(f'Inspecting {spec.name} ({spec.hf_id})')
log.info(f'Family: {spec.family}')
log.info(f'Quantization: {USE_QUANTIZATION}')

# Resolve the HF class — this is the exact class the runner will dispatch.
hf_class = _resolve_hf_class(spec.family, spec.hf_id, trust_remote_code=spec.requires_trust_remote_code)
print(f'Resolved HF class: {hf_class.__name__}  (from {hf_class.__module__})')
log.info(f'HF class: {hf_class.__module__}.{hf_class.__name__}')

In [ ]:
# Actual model load.
t0 = time.time()
try:
    model = _load_model(spec, dtype=torch.float16, quantization=USE_QUANTIZATION, logger=log)
except Exception as e:
    log.error('Model load failed', e)
    raise
load_seconds = time.time() - t0
print(f'Loaded in {load_seconds:.1f}s')
print(f'Class:            {type(model).__name__}')
print(f'Model device:     {next(model.parameters()).device}')
print(f'First-param dtype:{next(model.parameters()).dtype}')
raw_total = sum(p.numel() for p in model.parameters())
deduped   = sum(p.numel() for _, p in {id(p): p for p in model.parameters()}.items())
print(f'Total params (sum p.numel()):    {raw_total:>15,}  ({raw_total/1e6:>9.2f} M)')
print(f'Unique params (deduped by id):   {deduped:>15,}  ({deduped/1e6:>9.2f} M)')
if raw_total != deduped:
    delta = raw_total - deduped
    print(f'  → {delta:,} parameters are SHARED across more than one tensor reference (weight-tying).')
    print(f'    The deduped figure is the methodologically correct headline count — that is what')
    print(f'    notebook 02 reports. Section 10 below identifies which weights are tied.')
log.info(f'load_seconds={load_seconds:.1f}')
log.info(f'raw_total={raw_total} deduped={deduped} delta={raw_total - deduped}')

---
## 5. Top-level architecture summary

First look at the model's `__repr__` (truncated), then the config dict, then a back-of-the-envelope fp16 footprint, then the `tie_word_embeddings` flag.

**What to look for:**
* The repr's outermost wrapper tells you which HF class is in play (e.g. `LlavaForConditionalGeneration`, `ChameleonForConditionalGeneration`).
* The config's `architectures` field is what HuggingFace dispatches on.
* `tie_word_embeddings = True` means `lm_head.weight` IS `embed_tokens.weight` (one tensor, two names); raw param sum will double-count those.
* The fp16 footprint is `unique_params × 2 bytes`. Activations are NOT included here — that's notebook 04.

In [ ]:
if SHOW_FULL_MODULE_TREE:
    tree_str = str(model)
    tree_lines = tree_str.splitlines()
    if len(tree_lines) > 2 * MODULE_TREE_LINES:
        head = '\n'.join(tree_lines[:MODULE_TREE_LINES])
        tail = '\n'.join(tree_lines[-MODULE_TREE_LINES:])
        print(head)
        print(f'\n... ({len(tree_lines) - 2 * MODULE_TREE_LINES} of {len(tree_lines)} lines truncated) ...\n')
        print(tail)
    else:
        print(tree_str)
else:
    print('(SHOW_FULL_MODULE_TREE=False — skipping; set True to see full str(model).)')

In [ ]:
# Config dict — what HuggingFace knows about this model
cfg = getattr(model, 'config', None)
if cfg is None:
    print('Model has no .config attribute (very unusual).')
else:
    cfg_dict = cfg.to_dict() if hasattr(cfg, 'to_dict') else cfg.__dict__
    interesting_keys = [
        'architectures', 'model_type', 'hidden_size', 'num_hidden_layers',
        'num_attention_heads', 'intermediate_size', 'vocab_size',
        'image_size', 'patch_size', 'num_image_tokens',
        'tie_word_embeddings', 'torch_dtype',
    ]
    print('Selected config fields:')
    for k in interesting_keys:
        if k in cfg_dict:
            print(f'  {k:30s} = {cfg_dict[k]!r}')
    # Vision sub-config (common in multimodal models)
    vision_cfg = cfg_dict.get('vision_config') or cfg_dict.get('vision_tower_config')
    if isinstance(vision_cfg, dict):
        print("\n  vision_config:")
        for k in ['hidden_size', 'num_hidden_layers', 'patch_size', 'image_size', 'num_attention_heads']:
            if k in vision_cfg:
                print(f'    {k:28s} = {vision_cfg[k]!r}')
    # Text/language sub-config
    text_cfg = cfg_dict.get('text_config') or cfg_dict.get('language_config')
    if isinstance(text_cfg, dict):
        print("\n  text_config:")
        for k in ['hidden_size', 'num_hidden_layers', 'num_attention_heads', 'vocab_size', 'tie_word_embeddings']:
            if k in text_cfg:
                print(f'    {k:28s} = {text_cfg[k]!r}')

In [ ]:
# fp16 footprint approximation (unique_params × 2 bytes; activations excluded)
fp16_bytes = deduped * 2
fp32_bytes = deduped * 4
print(f'fp16 footprint (weights only): {fp16_bytes / 1e9:.2f} GB')
print(f'fp32 footprint (weights only): {fp32_bytes / 1e9:.2f} GB')
print(f'Spec-declared fp16_GB hint:     {spec.fits_in_gb_fp16:.2f} GB  (peak load + tokenizer + buffers; '
      'usually 20-50% larger than the weights-only number)')

# tie_word_embeddings flag
tie_flag = getattr(getattr(model, 'config', None), 'tie_word_embeddings', None)
if tie_flag is None:
    # Try the text/language sub-config
    text_cfg = getattr(getattr(model, 'config', None), 'text_config', None)
    if text_cfg is not None:
        tie_flag = getattr(text_cfg, 'tie_word_embeddings', None)
print(f'config.tie_word_embeddings:    {tie_flag}')
if tie_flag is True:
    print('  → lm_head.weight is expected to share storage with the input embedding table.')
    print('    §10 below verifies this by comparing data_ptr()s.')

---
## 6. Top-level children + named_modules() walk

`model.named_children()` shows only the direct submodules (one level down) — this tells you where to look for V/F/L/H. `model.named_modules()` shows every module at every depth — useful when the V/F/L/H modules are nested.

The walk is capped at `MAX_NAMED_MODULES` rows so the cell doesn't explode for 100+ layer LLMs. The full walk (uncapped) is appended to the inspection .txt report at the end.

In [ ]:
print('Top-level children of model:')
for name, child in model.named_children():
    n_direct = sum(p.numel() for p in child.parameters(recurse=False))
    n_recur  = sum(p.numel() for p in child.parameters())
    print(f'  model.{name:30s}  type={type(child).__name__:35s}  direct={n_direct:>12,}  recursive={n_recur:>14,}')

# If the model wraps an inner .model (common HF pattern: ForConditionalGeneration → Model)
if hasattr(model, 'model') and not isinstance(model.model, torch.nn.Linear):
    print('\nTop-level children of model.model (inner wrapper):')
    try:
        for name, child in model.model.named_children():
            n_direct = sum(p.numel() for p in child.parameters(recurse=False))
            n_recur  = sum(p.numel() for p in child.parameters())
            print(f'  model.model.{name:25s}  type={type(child).__name__:35s}  direct={n_direct:>12,}  recursive={n_recur:>14,}')
    except Exception as e:
        print(f'  (could not walk model.model: {type(e).__name__}: {e})')

In [ ]:
# Filtered named_modules() walk — leave FILTER_PATTERN='' for everything (up to MAX_NAMED_MODULES).
FILTER_PATTERN = ''   # e.g. 'vision', 'language', 'lm_head', 'embed_tokens', 'experts'

rows = []
n_total_modules = 0
for i, (name, mod) in enumerate(model.named_modules()):
    n_total_modules += 1
    if FILTER_PATTERN and FILTER_PATTERN.lower() not in name.lower():
        continue
    if len(rows) >= MAX_NAMED_MODULES:
        continue
    direct = sum(p.numel() for p in mod.parameters(recurse=False))
    recur  = sum(p.numel() for p in mod.parameters())
    rows.append({
        'depth':     name.count('.'),
        'name':      name or '<root>',
        'type':      type(mod).__name__,
        'direct':    direct,
        'recursive': recur,
    })

mod_df = pd.DataFrame(rows)
print(f'Model has {n_total_modules:,} modules in total (depth-first).')
print(f'Showing {len(mod_df)} (filter={FILTER_PATTERN!r}, capped at MAX_NAMED_MODULES={MAX_NAMED_MODULES})')
mod_df

---
## 7. Run `extract_components` and verify the schema

This is the moment of truth — does the extractor for this family find each component? For the standard V/F/L/H schema, all four keys should be present (though one or two may legitimately be `None`, e.g. CLIP has no `head`). For native-fusion (Chameleon) you should see `fused_transformer` + `embeddings` populated and V/F/L all `None`. For decoupled-vision (Janus) you should see both `vision_und` and `vision_gen` populated.

**Red flags:**
* A component reports 0 total params but is not `None` → the extractor returned an empty container; the real submodule is somewhere else.
* All four V/F/L/H are `None` but family is NOT `native_early_fusion` / `native_decoupled_encoder` → the extractor branch for this family is broken.
* `fused_transformer` is present for a non-Chameleon model → the wrong extractor branch fired.

In [ ]:
# Extract components using the family-dispatched extractor.
comps = extract_components(model, spec.family)
print(f'Family: {spec.family}')
print(f'Schema keys returned by extractor: {list(comps.keys())}')
print()
for cname, mod in comps.items():
    if mod is None:
        print(f'  {cname:20s}: None  (this is OK if the architecture genuinely lacks this component, '
              'otherwise it is a BUG in the extractor)')
    else:
        print(f'  {cname:20s}: {type(mod).__name__:30s}  (will be measured)')
log.section('Extracted components')
for cname, mod in comps.items():
    log.info(f'{cname}: {type(mod).__name__ if mod is not None else "None"}')

---
## 8. Per-component parameter walk

For each non-None component, print:
* total parameters (`count_params`)
* trainable parameters under the canonical freezing recipe (`compute_trainable_from_policy`)
* the top-level children of that component with their own param counts (depth-1 breakdown so you can see e.g. how many transformer blocks the vision tower has)

If the recipe says a component should be frozen, its `trainable_M` will be 0 in the DataFrame — that is the F-5 fix in action.

In [ ]:
# Apply freezing policy and build a per-component summary.
trainable_map = compute_trainable_from_policy(spec, comps)
policy_map    = trainable_components_for_family(spec.family)

ext_rows = []
for cname, mod in comps.items():
    if mod is None:
        ext_rows.append({
            'component':       cname,
            'type':            'None',
            'params_total':    0,
            'params_total_M':  0.0,
            'trainable':       0,
            'trainable_M':     0.0,
            'trainable_pct':   0.0,
            'policy_trainable': bool(policy_map.get(cname, True)),
        })
        continue
    total = count_params(mod)
    train = trainable_map.get(cname, 0)
    ext_rows.append({
        'component':       cname,
        'type':            type(mod).__name__,
        'params_total':    total,
        'params_total_M':  round(total / 1e6, 2),
        'trainable':       train,
        'trainable_M':     round(train / 1e6, 2),
        'trainable_pct':   round(100 * train / max(total, 1), 1),
        'policy_trainable': bool(policy_map.get(cname, True)),
    })
ext_df = pd.DataFrame(ext_rows)
ext_df

In [ ]:
# Depth-1 breakdown for each non-None component — what immediate children does it have?
for cname, mod in comps.items():
    if mod is None:
        continue
    total = count_params(mod)
    print(f'\n=== {cname}  ({type(mod).__name__})  total={total:,} ({total/1e6:.2f} M) ===')
    try:
        children = list(mod.named_children())
    except Exception as e:
        print(f'  (could not enumerate children: {type(e).__name__}: {e})')
        continue
    if not children:
        print(f'  (leaf module — no submodules)')
        continue
    for child_name, child in children:
        n_direct = sum(p.numel() for p in child.parameters(recurse=False))
        n_recur  = sum(p.numel() for p in child.parameters())
        pct = 100 * n_recur / max(total, 1)
        print(f'  {child_name:30s} {type(child).__name__:30s} direct={n_direct:>12,} recursive={n_recur:>14,} ({pct:>5.1f}%)')

---
## 9. Verify freezing policy vs `requires_grad`

On a fresh `from_pretrained` load, every parameter has `requires_grad=True` regardless of the model's intended training recipe. That's why Phase-1 F5 was degenerate — counting trainables via `requires_grad` reported 100% trainable for IDEFICS (which actually trains only 3.4%).

This cell shows both numbers side-by-side so you can see the gap. The `params_trainable` shown in §8 (from the policy) is the methodologically correct headline.

In [ ]:
rows = []
for cname, mod in comps.items():
    if mod is None:
        continue
    total = count_params(mod)
    grad_true = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    policy_train = trainable_map.get(cname, 0)
    rows.append({
        'component':            cname,
        'total_M':              round(total / 1e6, 2),
        'requires_grad_M':      round(grad_true / 1e6, 2),    # naive — always == total after from_pretrained
        'policy_trainable_M':   round(policy_train / 1e6, 2), # F-5 corrected
        'requires_grad_pct':    round(100 * grad_true / max(total, 1), 1),
        'policy_trainable_pct': round(100 * policy_train / max(total, 1), 1),
    })
freeze_df = pd.DataFrame(rows)
print('Left column (requires_grad_M) is the WRONG answer — it is what you would get without the F-5 fix.')
print('Right column (policy_trainable_M) is what notebook 02 reports.')
freeze_df

---
## 10. Layer-by-layer leaf-module table

Flatten every leaf module (no children) into a DataFrame: path, type, num_params, dtype. Sorted by params descending so the heavy hitters are at the top.

This is the most useful single view for understanding where the params actually live. For an LLM, you'll see ~32 `Linear` layers in each transformer block (q/k/v/o + mlp); for a ViT you'll see ~24 `Linear` layers (q/k/v/proj + mlp fc1/fc2).

The full table is saved to the .txt inspection report; only the top `LEAF_TABLE_TOP_N` rows are shown inline.

In [ ]:
leaf_rows = []
for name, mod in model.named_modules():
    # Leaf = no children
    if list(mod.children()):
        continue
    n_params = sum(p.numel() for p in mod.parameters(recurse=False))
    # Detect dtype of the largest direct parameter (or the first one if all empty)
    dtype_str = '-'
    direct_params = list(mod.parameters(recurse=False))
    if direct_params:
        dtype_str = str(direct_params[0].dtype)
    leaf_rows.append({
        'module_path': name or '<root>',
        'module_type': type(mod).__name__,
        'num_params':  n_params,
        'num_params_M': round(n_params / 1e6, 3),
        'dtype':       dtype_str,
    })

leaf_df_full = pd.DataFrame(leaf_rows).sort_values('num_params', ascending=False).reset_index(drop=True)
n_leaves = len(leaf_df_full)
n_param_leaves = (leaf_df_full['num_params'] > 0).sum()
print(f'Total leaf modules: {n_leaves:,}  ({n_param_leaves:,} carry parameters)')
print(f'Sum of all leaf params: {leaf_df_full["num_params"].sum():,}  (should match the deduped total above when no tying)')
print(f'Top {LEAF_TABLE_TOP_N} leaves by param count:')
leaf_df_full.head(LEAF_TABLE_TOP_N)

In [ ]:
# Layer-type histogram — useful for sanity-checking the structure
type_hist = (
    leaf_df_full.groupby('module_type')
    .agg(n_layers=('num_params', 'size'), total_params=('num_params', 'sum'))
    .sort_values('total_params', ascending=False)
)
type_hist['total_params_M'] = (type_hist['total_params'] / 1e6).round(2)
type_hist['pct_of_model'] = (100 * type_hist['total_params'] / max(type_hist['total_params'].sum(), 1)).round(1)
print('Param mass by layer type (leaf modules only):')
type_hist

---
## 11. Tied-weight detection (LLaVA-1.5 152M investigation tool)

Group parameters by `id(p.data)` — parameters sharing the same storage are the same physical tensor under two names. This is how PyTorch implements weight-tying.

**What you'll see for tied models** (LLaMA-family, Mistral, Qwen2+, etc.):
* The `lm_head.weight` group will contain TWO names: `lm_head.weight` and `model.language_model.embed_tokens.weight` (or similar).
* The raw `sum(p.numel())` over-counts that weight by `vocab_size × hidden_size`.
* This is the entire explanation for the LLaVA-1.5 `~152M` discrepancy between Sunita-1 cell 9 (raw) and Phase-2 (deduped).

Inversely, for CLIP and SigLIP you should see NO ties — these models train their text and vision projections independently.

In [ ]:
# Group parameters by storage id; report only the tied groups (size > 1).
groups = {}
for pname, p in model.named_parameters():
    groups.setdefault(p.data_ptr(), []).append((pname, p.numel(), tuple(p.shape)))

tied_groups = [g for g in groups.values() if len(g) > 1]
n_tied_params = sum(g[0][1] for g in tied_groups)  # each tied tensor only counted once

print(f'Total parameter NAMES (named_parameters()): {sum(len(g) for g in groups.values()):,}')
print(f'Total UNIQUE parameter STORAGES:            {len(groups):,}')
print(f'Number of TIED groups (>1 name per storage): {len(tied_groups)}')
print(f'Params represented by tied storages: {n_tied_params:,} ({n_tied_params/1e6:.2f} M)')
print(f'  (this many params would be over-counted if you summed named_parameters() without dedup)')
print()

if not tied_groups:
    print('No weight-tying detected — raw and deduped totals should match exactly.')
else:
    print('Tied groups detected:')
    for i, group in enumerate(tied_groups):
        numel = group[0][1]
        shape = group[0][2]
        print(f'  [{i+1}] storage of shape {shape} = {numel:,} params, shared across:')
        for pname, _n, _s in group:
            print(f'        {pname}')
    print()
    print('Verdict: if the SAME storage backs both lm_head.weight and embed_tokens.weight,')
    print('the Sunita-1 / Phase-1 raw total over-counted by exactly that storage size.')
    print('Notebook 02 uses count_params(component), which iterates parameters() (does NOT dedup).')
    print('Components that contain BOTH tied names will double-count; components that contain only ONE will not.')

---
## 12. `named_parameters()` head/tail dump

First N and last N entries from `named_parameters()` with shapes. Useful for:
* Spotting unusual layer orderings (e.g. a vision tower buried inside the language model).
* Confirming the final `lm_head` weight is where you expect it.
* Catching `lm_head.weight` appearing twice (tied).

In [ ]:
all_params = list(model.named_parameters())
n = len(all_params)
print(f'Total named parameters: {n:,}')
print(f'\n=== First {PARAM_DUMP_HEAD_TAIL} entries ===')
for pname, p in all_params[:PARAM_DUMP_HEAD_TAIL]:
    print(f'  {pname:80s} {tuple(p.shape)!s:25s} {p.numel():>14,}  {str(p.dtype):16s}')
if n > 2 * PARAM_DUMP_HEAD_TAIL:
    print(f'\n  ... ({n - 2*PARAM_DUMP_HEAD_TAIL} entries omitted) ...\n')
    print(f'=== Last {PARAM_DUMP_HEAD_TAIL} entries ===')
    for pname, p in all_params[-PARAM_DUMP_HEAD_TAIL:]:
        print(f'  {pname:80s} {tuple(p.shape)!s:25s} {p.numel():>14,}  {str(p.dtype):16s}')

---
## 13. Save the inspection report

Everything you've seen above (plus the full leaf-module table, full named_parameters dump, and full tied-group report) is written to a single `.txt` at `outputs/logs/nb01_inspect_<sanitized_hf_id>.txt`. Useful for attaching to a GitHub issue or pasting into a paper appendix.

In [ ]:
# Compose a comprehensive text report and write it alongside the log.
report_path = LOG_DIR / f'{INSPECT_LABEL}.report.txt'
buf = StringIO()
buf.write(f'=== Notebook 01 inspection report ===\n')
buf.write(f'Model:          {spec.name}  ({spec.hf_id})\n')
buf.write(f'Family:         {spec.family}\n')
buf.write(f'Year:           {spec.release_year}\n')
buf.write(f'Native HW:      {spec.native_image_hw}\n')
buf.write(f'Quantization:   {USE_QUANTIZATION}\n')
buf.write(f'Load seconds:   {load_seconds:.1f}\n')
buf.write(f'HF class used:  {hf_class.__module__}.{hf_class.__name__}\n')
buf.write(f'\n--- Parameter totals ---\n')
buf.write(f'raw_total (sum p.numel()):   {raw_total:,}\n')
buf.write(f'deduped (by id):             {deduped:,}\n')
buf.write(f'tied delta:                   {raw_total - deduped:,}\n')
buf.write(f'fp16 footprint (weights):    {deduped * 2 / 1e9:.2f} GB\n')
buf.write(f'config.tie_word_embeddings:  {tie_flag}\n')

buf.write(f'\n--- Per-component summary ---\n')
buf.write(ext_df.to_string(index=False) + '\n')

buf.write(f'\n--- Freezing-policy verification ---\n')
buf.write(freeze_df.to_string(index=False) + '\n')

buf.write(f'\n--- Layer-type histogram ---\n')
buf.write(type_hist.to_string() + '\n')

buf.write(f'\n--- Full leaf-module table ({len(leaf_df_full)} rows) ---\n')
buf.write(leaf_df_full.to_string(index=False) + '\n')

buf.write(f'\n--- Tied-weight groups ({len(tied_groups)} groups) ---\n')
if not tied_groups:
    buf.write('  (no tied weights detected)\n')
else:
    for i, group in enumerate(tied_groups):
        numel = group[0][1]; shape = group[0][2]
        buf.write(f'  [{i+1}] shape {shape} = {numel:,} params:\n')
        for pname, _n, _s in group:
            buf.write(f'        {pname}\n')

buf.write(f'\n--- Full named_parameters() dump ({n} entries) ---\n')
for pname, p in all_params:
    buf.write(f'  {pname:80s} {tuple(p.shape)!s:25s} {p.numel():>14,}  {str(p.dtype):16s}\n')

report_text = buf.getvalue()
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)
log.info(f'Inspection report written ({len(report_text):,} chars) to {report_path.name}')
print(f'Inspection report saved:')
print(f'  {report_path}')
print(f'  ({len(report_text):,} chars, {len(report_text.splitlines()):,} lines)')
print(f'\nLog file:')
log_files = sorted(LOG_DIR.glob(f'*{INSPECT_LABEL}*.txt'))
for lf in log_files:
    print(f'  {lf}')

---
## 14. Cleanup

Free up GPU memory so you can re-run with a different `MODEL_NAME` (just go back to §3, change the value, and rerun §4 onward — no kernel restart needed).

In [ ]:
try:
    del model
except NameError:
    pass
try:
    del comps
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
log.close()
free_after, _ = free_disk_gb()
print(f'Model unloaded. Disk free now: {free_after:.1f} GB')
if torch.cuda.is_available():
    print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB allocated')

---
## Next steps

* If §7 showed a component as `None` that shouldn't be (or a 0 where you expect millions) → the extractor for this family needs a fix. Look in `vlm_audit/components.py` for the family branch and add the missing attribute path. The IDEFICS-9B F-3 fix was discovered exactly this way.
* If §11 showed tied weights → flag in `MODELS_AND_EXPERIMENTS.md` so the paper either reports the deduped figure (recommended) or explicitly notes the over-count.
* If §10 showed an unexpected dtype (e.g. fp32 where you asked for fp16) → check `_load_model` in `vlm_audit/runner.py`; the `dtype=` / `torch_dtype=` kwarg may not be passing through correctly for this HF class.
* Re-run §3 onward with a different `MODEL_NAME` to compare two extractors back-to-back. No kernel restart needed.
* When you've verified an extractor works for a model, you can run the full pipeline (`02_parameter_analysis_v3.ipynb`) with confidence.

*End of notebook 01 (verbose v2 — 2026-05-22). This notebook is for inspection only; no measurement JSONs are written.*